# Train a model, save it, use it

We train a straight-line regression once, save it to a file, and then use the saved model in an app.

The data here is **made up**, to show the steps. A real business would use its own sales records.

In [7]:
import numpy as np
import pandas as pd
import plotly.express as px
from sklearn.linear_model import LinearRegression
import pickle

## Made-up data

Two hundred days. Hotter days sell more ice cream, with some random noise.

In [8]:
# Generate sample data
np.random.seed(42)
n = 200

temperature = np.random.uniform(50, 100, n)
is_weekend = np.random.randint(0, 2, n)

sales = (
    5 * temperature
    + 30 * is_weekend
    - 100
    + np.random.normal(0, 25, n)
)

df = pd.DataFrame({
    "temperature": temperature,
    "is_weekend": is_weekend,
    "sales": sales.clip(0)
})

df.head()


,temperature,is_weekend,sales
0,68.727006,1,274.774326
1,97.535715,1,401.388568
2,86.599697,0,386.597088
3,79.932924,0,315.512597
4,57.800932,1,168.376095


In [9]:
# Visualize sales versus temperature
fig = px.scatter(
    df,
    x="temperature",
    y="sales",
    color="is_weekend",
    title="Ice Cream Sales versus Temperature"
)
fig.show()

## Fit the line

In [10]:
# Train the model using both predictors
x = df[["temperature", "is_weekend"]]
y = df["sales"]

model = LinearRegression()
model.fit(x, y)

print("Coefficients:", model.coef_)
print("Intercept:", round(model.intercept_, 1))
print("R2:", round(model.score(x, y), 2))

Coefficients: [ 5.11354458 28.56368849]
Intercept: -106.2
R2: 0.91


In [12]:
# Plot predictions for weekdays and weekends
temperatures = np.linspace(
    df["temperature"].min(),
    df["temperature"].max(),
    100
)

for weekend in [0, 1]:
    plot_data = pd.DataFrame({
        "temperature": temperatures,
        "is_weekend": weekend
    })

    predictions = model.predict(plot_data)

    fig.add_scatter(
        x=temperatures,
        y=predictions,
        mode="lines",
        name="Weekend" if weekend == 1 else "Weekday"
    )

fig.show()

# Test a prediction
new_day = pd.DataFrame({
    "temperature": [85],
    "is_weekend": [1]
})

print("Predicted sales:", round(model.predict(new_day)[0]))


Predicted sales: 357


Each extra degree adds about five cones. Now a prediction for a day with a high of 85 °F. The model wants a table with the same column name it was trained on.

In [6]:
new_day = pd.DataFrame({
    "temperature": [85],
    "is_weekend": [1]
})

print(round(model.predict(new_day)[0]))

357


## Save it

`pickle` writes the trained model to a file. We train once and use the file as many times as we like.

In [13]:
with open("model.pkl", "wb") as f:
    pickle.dump(model, f)

## Load it back and check that it works

This is exactly what the app will do.

In [14]:
with open("model.pkl", "rb") as f:
    loaded_model = pickle.load(f)

print("Loaded model prediction:", round(loaded_model.predict(new_day)[0]))   # the same number as before

Loaded model prediction: 357


## Download it

The file has to go in your GitHub repository, next to `app.py`.

In [15]:
from google.colab import files
files.download("model.pkl")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [17]:
!pip install -q streamlit


In [18]:
%%writefile app.py

import pickle
from datetime import date, timedelta

import pandas as pd
import requests
import streamlit as st

CITIES = {
    "Pittsburgh": (40.44, -79.99),
    "New York": (40.71, -74.01),
    "Chicago": (41.88, -87.63),
    "Miami": (25.76, -80.19),
    "Los Angeles": (34.05, -118.24),
}

LOW_TEMP, HIGH_TEMP = 50, 100
LOW_SALES, HIGH_SALES = 200, 300
URL = "https://api.open-meteo.com/v1/forecast"

st.set_page_config(
    page_title="Ice Cream Sales Predictor",
    page_icon="🍦"
)

with open("model.pkl", "rb") as f:
    model = pickle.load(f)

st.title("🍦 Ice Cream Sales Predictor")
st.caption(
    "Pick a city and a date. The app gets the weather forecast "
    "and predicts how many cones you will sell."
)

city = st.selectbox("City", list(CITIES))

selected_date = st.date_input(
    "Date",
    value=date.today() + timedelta(days=1),
    min_value=date.today(),
    max_value=date.today() + timedelta(days=16),
)

if st.button("Predict sales", type="primary"):

    # Get the weather forecast
    latitude, longitude = CITIES[city]

    response = requests.get(
        URL,
        params={
            "latitude": latitude,
            "longitude": longitude,
            "daily": "temperature_2m_max",
            "temperature_unit": "fahrenheit",
            "timezone": "auto",
            "start_date": str(selected_date),
            "end_date": str(selected_date),
        },
        timeout=30,
    )

    response.raise_for_status()

    temperature = response.json()["daily"]["temperature_2m_max"][0]

    # Determine whether the date is a weekend
    is_weekend = int(selected_date.weekday() >= 5)

    # Predict sales using both predictors
    features = pd.DataFrame({
        "temperature": [temperature],
        "is_weekend": [is_weekend]
    })

    prediction = max(0, model.predict(features)[0])

    # Display results
    col1, col2, col3 = st.columns(3)

    col1.metric("Forecast high", f"{temperature:.0f} °F")
    col2.metric(
        "Day type",
        "Weekend" if is_weekend else "Weekday"
    )
    col3.metric("Predicted sales", f"{prediction:.0f} cones")

    if temperature < LOW_TEMP or temperature > HIGH_TEMP:
        st.warning(
            f"This temperature is outside the range the model learned "
            f"from ({LOW_TEMP} to {HIGH_TEMP} °F). Treat the number with caution."
        )

    if prediction > HIGH_SALES:
        st.info("Busy day: make sure you are fully stocked.")
    elif prediction > LOW_SALES:
        st.info("A normal day is expected.")
    else:
        st.info("A slow day: consider a promotion, or order less.")

Writing app.py


In [19]:
!streamlit run app.py --server.port 8501 --server.address 0.0.0.0 > /content/streamlit.log 2>&1 &

In [20]:
from google.colab.output import eval_js
print(eval_js("google.colab.kernel.proxyPort(8501)"))

https://8501-m-s-usc1b9-3e9dzyt39hojl-b.us-central1.prod.r.colab.dev


In [21]:
import os

print(os.listdir("/content"))

['.config', 'streamlit.log', 'model.pkl', 'app.py', 'sample_data']


In [22]:
test_days = pd.DataFrame({
    "temperature": [60, 70, 80, 90, 85, 85],
    "is_weekend": [0, 0, 0, 0, 0, 1]
})

test_days["predicted_sales"] = model.predict(test_days)

print(test_days.round(1))

   temperature  is_weekend  predicted_sales
0           60           0            200.7
1           70           0            251.8
2           80           0            302.9
3           90           0            354.1
4           85           0            328.5
5           85           1            357.1
